In [3]:
import tensorflow as tf
import numpy as np
import cv2
import os 
from tensorflow.keras import layers, Model

In [10]:
def conv_block(x,growth_rate):
    x1=layers.BatchNormalization()(x)
    x1=layers.ReLU()(x1)
    x1=layers.Conv2D(4 * growth_rate , (1,1), padding='same', use_bias=False)(x1)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.ReLU()(x1)
    x1 = layers.Conv2D(growth_rate, (3,3), padding='same', use_bias=False)(x1)  # 3x3 Conv
    x = layers.Concatenate()([x, x1])
    return x


In [11]:
def dense_block(x, num_layers, growth_rate):
    for _ in range(num_layers):
        x = conv_block(x, growth_rate)
    return x

In [12]:
def transition_layer(x, reduction):
    """Reduces the number of feature maps using 1x1 convolution and pooling."""
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(int(tf.keras.backend.int_shape(x)[-1] * reduction), (1,1), padding='same', use_bias=False)(x)
    x = layers.AveragePooling2D((2,2), strides=2)(x)
    return x

In [13]:
def DenseNet(input_shape=(224, 224, 3), num_classes=10, growth_rate=32, blocks=[6,12,24,16]):
    """Creates a DenseNet model."""
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, (7,7), strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((3,3), strides=2, padding='same')(x)

    # Dense Blocks and Transition Layers
    for i, num_layers in enumerate(blocks):
        x = dense_block(x, num_layers, growth_rate)
        if i != len(blocks) - 1:  # No transition after last dense block
            x = transition_layer(x, 0.5)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model

In [15]:
model=DenseNet()
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_121 (Conv2D)            (None, 112, 112, 64  9408        ['input_3[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization_121 (Batch  (None, 112, 112, 64  256        ['conv2d_121[0][0]']             
 Normalization)                 )                                                           